우리가 지금까지 진행한 방식 기준에서 **다음 단계**는 이 순서에요👇

1. Full 모델들 준비 (RF, LGBM, ET 등)
2. StackingRegressor로 메타 모델 구성
3. Auto-weight Voting / Stacking 두 가지 옵션 가능
4. predict_test, save_submission 재활용

---

---

# 📌 어디에 붙여야 하나?

* 팀원이 공유할 때는 `stacking_models.py`나 `ensemble_utils.py` 파일로 따로 저장
* analyzer 클래스 바깥에서 호출 가능
* PyCaret이 아니라 **순수 sklearn stacking**을 쓰기 때문에 모델 타입 차이에 강함
* 팀원 vectorizer 방식 다 달라도 `train_vectorized`가 존재하면 문제 없음

---

# 🧠 stacking 메타모델 추천 (meta_model)

쉬운 순서:

1. Ridge
2. LightGBM
3. CatBoost (GPU 사용 시)

---




In [ ]:
# ---------------------------------
# stacking_models.py 형태로 저장 추천!
# ---------------------------------

from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.linear_model import Ridge
import joblib
import numpy as np

def load_saved_models(model_paths):
    """
    model_paths: [list of pkl paths]
    """
    models = {}
    for p in model_paths:
        name = p.split("/")[-1].replace(".pkl", "")
        models[name] = joblib.load(p)
    return models


def build_stacking_model(models_dict, meta_model=None):
    """
    models_dict: {"rf": rf_model_object, "lgbm": lgbm_model_object ...}
    meta_model: default Ridge (linear) or any regressor
    """
    if meta_model is None:
        meta_model = Ridge(alpha=1.0)

    estimators = [(k, v) for k, v in models_dict.items()]

    stack_model = StackingRegressor(
        estimators=estimators,
        final_estimator=meta_model,
        passthrough=True,
        n_jobs=-1
    )
    return stack_model


def fit_stacking(stack_model, X, y):
    """
    Fit stacking model using given X,y
    """
    stack_model.fit(X, y)
    return stack_model


def save_stacking_model(stack_model, path):
    joblib.dump(stack_model, path)
    print(f"💾 stacking model saved: {path}")

In [ ]:
# step1: 후보 모델 로드 아래에 각자 저장된 모델명 넣기
model_paths = [
    "../models/full_rf_2025xxxx.pkl",
    "../models/full_lightgbm_2025xxxx.pkl",
    "../models/full_et_2025xxxx.pkl"
]

models_dict = load_saved_models(model_paths)

# step2: stacking 정립
stack_model = build_stacking_model(models_dict)

# step3: fit
X = analyzer.train_vectorized
y = analyzer.train['price']     # log1p 상태 그대로 OK
stack_model = fit_stacking(stack_model, X, y)

# step4: 저장
save_stacking_model(stack_model, "../models/full_stacked.pkl")


In [ ]:
# 예측 (test)
stack_preds = stack_model.predict(analyzer.test_vectorized)

In [ ]:
# 🔥 오토 가중치(soft voting) 버전 간단 예시
from sklearn.ensemble import VotingRegressor

vote_model = VotingRegressor(
    estimators=[(k, v) for k, v in models_dict.items()],
    weights=None    # 나중에 자동으로 튜닝할 수 있음
)
vote_model.fit(X, y)
joblib.dump(vote_model, "../models/full_voting.pkl")